In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [2]:
DATA_PATH = "../data/telco_customer_churn.csv"

df = pd.read_csv(DATA_PATH)

In [3]:
new_customer_missing = (
    df["TotalCharges"].str.strip().eq("")
    & df["tenure"].eq(0)
)

df.loc[
    new_customer_missing,
    "TotalCharges"
] = "0"

In [4]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [5]:
X = df.drop(
    columns=["customerID", "Churn"]
)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [7]:
def add_selected_features(
    data,
    selected_features
):
    result = data.copy()

    if "TenureGroup" in selected_features:
        result["TenureGroup"] = pd.cut(
            result["tenure"],
            bins=[
                -np.inf,
                12,
                24,
                48,
                np.inf
            ],
            labels=[
                "0-12",
                "13-24",
                "25-48",
                "49+"
            ]
        )

    if "TotalServices" in selected_features:
        service_columns = [
            "PhoneService",
            "MultipleLines",
            "OnlineSecurity",
            "OnlineBackup",
            "DeviceProtection",
            "TechSupport",
            "StreamingTV",
            "StreamingMovies"
        ]

        result["TotalServices"] = (
            result[service_columns]
            .eq("Yes")
            .sum(axis=1)
            + result["InternetService"]
            .ne("No")
            .astype(int)
        )

    if "AvgMonthlySpend" in selected_features:
        safe_tenure = result["tenure"].replace(
            0,
            np.nan
        )

        result["AvgMonthlySpend"] = (
            result["TotalCharges"]
            .div(safe_tenure)
            .fillna(0)
        )

    if "ContractPayment" in selected_features:
        result["ContractPayment"] = (
            result["Contract"].astype(str)
            + " | "
            + result["PaymentMethod"].astype(str)
        )

    if "SupportServicesCount" in selected_features:
        support_columns = [
            "OnlineSecurity",
            "DeviceProtection",
            "TechSupport"
        ]

        result["SupportServicesCount"] = (
            result[support_columns]
            .eq("Yes")
            .sum(axis=1)
        )

    return result

In [8]:
all_feature_names = [
    "TenureGroup",
    "TotalServices",
    "AvgMonthlySpend",
    "ContractPayment",
    "SupportServicesCount"
]

X_train_all_features = add_selected_features(
    X_train,
    all_feature_names
)

In [9]:
X_train_all_features[
    [
        "tenure",
        "TenureGroup",
        "TotalServices",
        "AvgMonthlySpend",
        "ContractPayment",
        "SupportServicesCount"
    ]
].head(10)

,tenure,TenureGroup,TotalServices,AvgMonthlySpend,ContractPayment,SupportServicesCount
3738,35,25-48,4,48.618571,Month-to-month | Electronic check,1
3151,15,13-24,3,76.770000,Month-to-month | Mailed check,1
4860,13,13-24,4,45.411538,Two year | Mailed check,2
3867,26,25-48,6,73.296154,Two year | Credit card (automatic),1
3810,1,0-12,2,44.550000,Month-to-month | Electronic check,0
2666,66,49+,7,78.728788,Two year | Credit card (automatic),2
2645,1,0-12,1,19.800000,Month-to-month | Credit card (automatic),0
189,40,25-48,4,59.477500,Month-to-month | Credit card (automatic),0
5767,65,49+,4,58.587692,Two year | Credit card (automatic),1
2257,60,49+,7,80.784167,One year | Credit card (automatic),2


In [ ]:
base_numeric_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [ ]:
base_categorical_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [ ]:
feature_types = {
    "TenureGroup": "categorical",
    "TotalServices": "numeric",
    "AvgMonthlySpend": "numeric",
    "ContractPayment": "categorical",
    "SupportServicesCount": "numeric"
}

In [ ]:
def create_model_pipeline(
    selected_features
):
    numeric_columns = (
        base_numeric_columns.copy()
    )

    categorical_columns = (
        base_categorical_columns.copy()
    )

    for feature in selected_features:
        if feature_types[feature] == "numeric":
            numeric_columns.append(feature)
        else:
            categorical_columns.append(feature)

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        ]
    )

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=1000
                )
            )
        ]
    )

    return model_pipeline